# ACE-Net Stage-1 — FV-LiteNet (visual) on MELD  ·  PERSON C

Trains ONE unimodal emotion extractor to the paper spec (**50 epochs, early-stop
patience 25**) on a Colab T4. Produces `stage1_visual_meld.pt`.

MELD uses a stratified-by-emotion split (paper protocol).

Set Runtime → **T4 GPU** before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Clone repo (feature branch)

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git checkout feat/acenet-training-pipeline
!git log --oneline -1

## 2. Install dependencies

In [ ]:
!pip -q install torch torchvision torchaudio transformers librosa pillow
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 3. Mount Drive + unzip data

Upload **`meld.zip`** to your Drive root. It must contain `MELD/train`, `MELD/dev`, `MELD/test` and the manifests.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, glob, shutil
ZIP = '/content/drive/MyDrive/meld.zip'   # adjust if needed
DST = '/content/Baseline_Training/data'
os.makedirs(DST, exist_ok=True)
with zipfile.ZipFile(ZIP) as z: z.extractall(DST)

# auto-relocate: ensure data/CREMA-D and data/MELD sit at the right level
for target in ['CREMA-D','MELD']:
    hits=[d for d in glob.glob(f'{DST}/**/{target}', recursive=True) if os.path.isdir(d)]
    for h in hits:
        want=os.path.join(DST,target)
        if os.path.abspath(h)!=os.path.abspath(want):
            shutil.move(h, want)
    # also pull loose CREMA genuine dirs under CREMA-D/
for sub in ['GENUINE_LastHalf','GENUINE_FirstHalf']:
    loose=os.path.join(DST,sub)
    if os.path.isdir(loose):
        os.makedirs(os.path.join(DST,'CREMA-D'),exist_ok=True)
        shutil.move(loose, os.path.join(DST,'CREMA-D',sub))
print('data/ ->', os.listdir(DST))
if os.path.isdir(f'{DST}/CREMA-D'): print('CREMA-D ->', os.listdir(f'{DST}/CREMA-D'))

## 4. Train (50 epochs, patience 25, augmentation on)

In [ ]:
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.train_stage1 --branch visual --dataset meld --batch-size 32 --epochs 50 --early-stop 25 --num-workers 2

## 5. Back up checkpoint to Drive (Colab is ephemeral!)

In [ ]:
import os, shutil
os.makedirs('/content/drive/MyDrive/acenet_ckpts', exist_ok=True)
for f in ['stage1_visual_meld.pt','stage1_visual_meld.log']:
    src=f'/content/Baseline_Training/checkpoints/{f}'
    if os.path.exists(src): shutil.copy(src, f'/content/drive/MyDrive/acenet_ckpts/{f}')
print('backed up:', os.listdir('/content/drive/MyDrive/acenet_ckpts'))

## 6. Evaluate (Accuracy, Weighted-F1, per-class F1, confusion)

In [ ]:
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.eval_stage1 --branch visual --dataset meld

## Hand-off

`stage1_visual_meld.pt` is now on Drive in `acenet_ckpts/`.
MELD checkpoints feed Tables 2/3 only; Stage-2 does not use them (no MELD forgeries).